# Module 03: Multi-Node JAX CPU Training & 10x Scale-Up
Because GPU/TPU quota can be difficult to acquire, this notebook provides a **100% accessible, zero-quota CPU multi-node path**.

Using `--xla_force_host_platform_device_count=4`, JAX simulates 4 virtual accelerator devices per CPU host while executing true inter-node multiprocess distribution across Kubernetes nodes!

### Key Concepts Covered:
1. **Pod Anti-Affinity**: Forcing Kubernetes to schedule worker pods on separate physical VM nodes (`topologyKey: "kubernetes.io/hostname"`).
2. **Headless Discovery & Batch Indexing**: Headless DNS discovery (`COORDINATOR_ADDRESS="jax-cpu-job-workers-0-0.jax-cpu-job:1234"`) and `batch.kubernetes.io/job-completion-index`.
3. **Mathematical Proof (`psum`)**: Validating gradient all-reduce synchronization mathematically.
   - For $N$ ranks with input $i+1$, expected sum is $\sum_{k=1}^N k = \frac{N(N+1)}{2}$.
   - 2 Nodes ($N=2$): Expected sum = **`3.0`** across 8 virtual devices.
   - 20 Nodes ($N=20$): Expected sum = **`210.0`** across 80 virtual devices.
4. **10x Scale-Up Experiment**: Resizing GKE cluster to 20 nodes and scaling JobSet to 20 pods in parallel.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import config
cfg = config.load_config("../config.env")

PROJECT_ID = cfg["PROJECT_ID"]
REGION = cfg["REGION"]
ZONE = cfg["ZONE"]
CLUSTER_NAME = cfg["CLUSTER_NAME"]
REPO = cfg["ARTIFACT_REGISTRY_REPO"]
CPU_IMAGE = cfg["CPU_IMAGE_NAME"]
TAG = cfg["IMAGE_TAG"]

CPU_FULL_IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO}/{CPU_IMAGE}:{TAG}"
print(f"Target CPU Image: {CPU_FULL_IMAGE}")

## PART A: 2-Node CPU Multi-Node Experiment (8 Virtual Devices)

### 1. Render 2-Node CPU JobSet Manifest

In [ ]:
manifest_path = Path("../manifests/jobset-cpu.yaml")
rendered_path = Path("../manifests/jobset-cpu-rendered.yaml")

with open(manifest_path, "r") as f:
    content = f.read()

content = content.replace("LOCATION-docker.pkg.dev/PROJECT_ID/ARTIFACT_REGISTRY_REPO/CPU_IMAGE_NAME:IMAGE_TAG", CPU_FULL_IMAGE)

with open(rendered_path, "w") as f:
    f.write(content)

print(f"Saved rendered JobSet manifest to {rendered_path}")

### 2. Deploy 2-Node JobSet & Verify Pod Anti-Affinity Placement

In [ ]:
!kubectl apply -f ../manifests/jobset-cpu-rendered.yaml

In [ ]:
import time
print("Waiting for worker pods to start...")
for _ in range(6):
    !kubectl get pods -l jobset.x-k8s.io/jobset-name=jax-cpu-job -o custom-columns=POD_NAME:.metadata.name,POD_IP:.status.podIP,NODE:.spec.nodeName,STATUS:.status.phase
    time.sleep(5)

### 3. Verify Headless DNS Service & Endpoints

In [ ]:
!kubectl get service jax-cpu-job
!kubectl get endpoints jax-cpu-job

### 4. Stream Logs & Check Mathematical Proof (Expected Sum: 3.0)

In [ ]:
!kubectl logs -l jobset.x-k8s.io/jobset-name=jax-cpu-job --all-containers --tail=100

---

## PART B: 10x Scale-Up Experiment (20 Nodes, 80 Virtual Devices)
Now we scale our cluster from 2 nodes to **20 nodes** to prove multi-node JAX scaling at 10x capacity!

### 1. Resize GKE Cluster to 20 Nodes

In [ ]:
!gcloud container clusters resize {CLUSTER_NAME} \
    --node-pool=default-pool \
    --num-nodes=20 \
    --zone={ZONE} \
    --quiet

In [ ]:
!kubectl get nodes -l cloud.google.com/gke-nodepool=default-pool --no-headers | wc -l

### 2. Render 20-Node JobSet Manifest

In [ ]:
scale_path = Path("../manifests/jobset-scale-20-rendered.yaml")

with open(manifest_path, "r") as f:
    scale_content = f.read()

scale_content = scale_content.replace("LOCATION-docker.pkg.dev/PROJECT_ID/ARTIFACT_REGISTRY_REPO/CPU_IMAGE_NAME:IMAGE_TAG", CPU_FULL_IMAGE)
scale_content = scale_content.replace("name: jax-cpu-job", "name: jax-scale-job")
scale_content = scale_content.replace('values: ["jax-cpu-job"]', 'values: ["jax-scale-job"]')
scale_content = scale_content.replace("parallelism: 2", "parallelism: 20")
scale_content = scale_content.replace("completions: 2", "completions: 20")
scale_content = scale_content.replace('value: "2"', 'value: "20"')
scale_content = scale_content.replace("jax-cpu-job-workers-0-0.jax-cpu-job", "jax-scale-job-workers-0-0.jax-scale-job")

with open(scale_path, "w") as f:
    f.write(scale_content)

print(f"Saved 20-Node Scale JobSet manifest to {scale_path}")

### 3. Deploy 20-Node JobSet

In [ ]:
!kubectl apply -f ../manifests/jobset-scale-20-rendered.yaml

In [ ]:
print("Monitoring 20 Pods scheduling across 20 Nodes...")
for _ in range(8):
    !kubectl get jobset jax-scale-job
    time.sleep(5)

### 4. Verify 20-Pod Physical Node Distribution

In [ ]:
!kubectl get pods -l jobset.x-k8s.io/jobset-name=jax-scale-job -o custom-columns=POD_NAME:.metadata.name,POD_IP:.status.podIP,NODE:.spec.nodeName,STATUS:.status.phase

### 5. Stream Cross-Node Logs & Validate Mathematical Sum (Expected Sum: 210.0)

In [ ]:
!kubectl logs -l jobset.x-k8s.io/jobset-name=jax-scale-job --all-containers=true --prefix=true --max-log-requests=25